In [7]:
from dotenv import load_dotenv

load_dotenv()

True

In [8]:
from langchain_openrouter import ChatOpenRouter
model = ChatOpenRouter(model="openai/gpt-5-nano")   # or anthropic/claude-3.5-haiku, etc.

In [10]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [12]:
print(uploader.value)

({'name': 'inside_fridge.PNG', 'type': 'image/png', 'size': 102709, 'content': <memory at 0x000001D8045DB400>, 'last_modified': datetime.datetime(2026, 8, 25, 9, 34, 46, 83000, tzinfo=datetime.timezone.utc)},)


In [13]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [14]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [17]:
from langgraph.checkpoint.memory import InMemorySaver  
from langchain.agents import create_agent
from langchain.messages import HumanMessage

from langchain_openrouter import ChatOpenRouter
model = ChatOpenRouter(model="openai/gpt-5-nano")   # or anthropic/claude-3.5-haiku, etc.
# system_prompt = """You are a cooking expert helping with recipe lookup based on available ingredients. 
# A list of ingredients will be given to you and you will find the recipes on the internet that 
# use them and present the ones that are possible to cook with the available ingredients."""
recipe_agent = create_agent(
    model=model,
    # system_prompt=system_prompt,
    tools=[web_search],
    checkpointer=InMemorySaver(),  
)

# question = HumanMessage(content="""Tell me a few recipes I can cook some food considering available ingredients are as below:Tomatoes, Potatoes, Meat, Eggs""")
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me a few recipes I can cook some food considering available ingredients are as you see in the image:"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])
config = {"configurable": {"thread_id": "1"}}

response = recipe_agent.invoke(
    {"messages": [multimodal_question]},
        config,  
)

print(response['messages'][1].content)

Nice! Based on the image, you’ve got:

- Grapes (green and red)
- Cherry tomatoes
- Apples
- Cucumbers
- Zucchini
- Peaches (or nectarines)

Here are a few quick recipes you can make with those, plus easy add-ins if you have staples (olive oil, lemon, salt, pepper, honey, yogurt, garlic, pasta, herbs).

1) Fresh fruit and grape salad
- Ingredients: a mix of grapes, chopped apples, chopped peaches
- Steps:
  - Wash and chop fruit into bite-sized pieces
  - Toss together in a bowl
  - Optional: squeeze a little lemon juice and drizzle honey; toss again
  - Chill 5–10 minutes and serve
- Why it’s great: no cooking needed, bright and refreshing

2) Cucumber-apple-grape salad
- Ingredients: 1 cucumber (sliced), 1 apple (sliced), a handful of grapes (halved)
- Steps:
  - Slice cucumber and apple; halve grapes
  - Whisk a quick dressing with olive oil, lemon juice, salt, and pepper
  - Toss everything with the dressing
- Optional add-ins: chopped mint or a pinch of chili flakes for a kick
- W

In [20]:
from pprint import pprint
question = HumanMessage(content="which one has more Protein?")

response = recipe_agent.invoke(
    {"messages": [question]},
    config,  
)

pprint(response)

{'messages': [HumanMessage(content=[{'type': 'text', 'text': 'Tell me a few recipes I can cook some food considering available ingredients are as you see in the image:'}, {'type': 'image', 'base64': '...', 'mime_type': 'image/png'}], additional_kwargs={}, response_metadata={}, id='1c1db152-02c4-4ca1-a2ca-f508cdddca54'),
              AIMessage(content='Nice! Based on the image, you’ve got:\n\n- Grapes (green and red)\n- Cherry tomatoes\n- Apples\n- Cucumbers\n- Zucchini\n- Peaches (or nectarines)\n\nHere are a few quick recipes you can make with those, plus easy add-ins if you have staples (olive oil, lemon, salt, pepper, honey, yogurt, garlic, pasta, herbs).\n\n1) Fresh fruit and grape salad\n- Ingredients: a mix of grapes, chopped apples, chopped peaches\n- Steps:\n  - Wash and chop fruit into bite-sized pieces\n  - Toss together in a bowl\n  - Optional: squeeze a little lemon juice and drizzle honey; toss again\n  - Chill 5–10 minutes and serve\n- Why it’s great: no cooking needed, 